In [ ]:
!curl -L -o amazon-fine-food-reviews.zip\
  https://www.kaggle.com/api/v1/datasets/download/snap/amazon-fine-food-reviews

!unzip amazon-fine-food-reviews.zip

In [73]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
import nltk
import nltk.corpus
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /Users/tudor/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [74]:
train = pd.read_csv("Reviews.csv")
train = train.head(1000)

In [75]:
train

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...
...,...,...,...,...,...,...,...,...,...,...
995,996,B006F2NYI2,A1D3F6UI1RTXO0,Swopes,1,1,5,1331856000,Hot & Flavorful,BLACK MARKET HOT SAUCE IS WONDERFUL.... My hus...
996,997,B006F2NYI2,AF50D40Y85TV3,Mike A.,1,1,5,1328140800,Great Hot Sauce and people who run it!,"Man what can i say, this salsa is the bomb!! i..."
997,998,B006F2NYI2,A3G313KLWDG3PW,kefka82,1,1,5,1324252800,this sauce is the shiznit,this sauce is so good with just about anything...
998,999,B006F2NYI2,A3NIDDT7E7JIFW,V. B. Brookshaw,1,2,1,1336089600,Not Hot,Not hot at all. Like the other low star review...


In [76]:
text = train["Text"].values

In [77]:
import string

pct = string.punctuation
stop = nltk.corpus.stopwords.words('english')

def proc_sent(sent):
    sent = sent.lower()
    for st in stop: sent = sent.replace("<br />", " ")
    for p in pct: sent = sent.replace(p, " ")
    for st in stop: sent = sent.replace(" " + st + " ", " ")

    return sent;

text = [proc_sent(s) for s in text]

In [78]:
X = []
y = []

In [79]:
text

['i bought several vitality canned dog food products found good quality  product looks like stew processed meat smells better  labrador finicky appreciates product better  ',
 'product arrived labeled jumbo salted peanuts   peanuts actually small sized unsalted  sure error vendor intended represent product  jumbo  ',
 'this confection around centuries   light  pillowy citrus gelatin nuts   case filberts  cut tiny squares liberally coated powdered sugar   tiny mouthful heaven   chewy  flavorful   highly recommend yummy treat   familiar story c  lewis   lion  witch  wardrobe    treat seduces edmund selling brother sisters witch ',
 'if looking secret ingredient robitussin believe found   got addition root beer extract ordered  good  made cherry soda   flavor medicinal ',
 'great taffy great price   wide assortment yummy taffy   delivery quick   taffy lover  deal ',
 'i got wild hair taffy ordered five pound bag  taffy enjoyable many flavors  watermelon  root beer  melon  peppermint  grap

In [80]:
tok = Tokenizer()

In [81]:
tok.fit_on_texts(text)
total_words = len(tok.word_index) + 1

In [82]:
tokenized_texts = []

for sent in text:
    words = [tok.word_index[w] for w in sent.split()]
    tokenized_texts.append(words)

In [83]:
import numpy as np

WINDOW = 7

def make_ds(data: list):
    X = []
    y = []
    for i in range(WINDOW, len(data), 1):
        X.append(data[i - WINDOW:i])
        y.append(data[i]);

    return np.array(X), np.array(y)

In [84]:
def creata_big_ds(data: list[list]):
    X = []
    y = []

    for pr in data:
        export = make_ds(pr) 
        X.append(export[0])
        y.append(export[1])
    
    X_ = []
    y_ = []

    for np_arr in X:
        for sample in np_arr:
            X_.append(sample)
    for np_arr in y:
        for sample in np_arr:
            y_.append(sample)

    
    return np.array(X_), np.array(y_)

In [85]:
X, y = creata_big_ds(tokenized_texts)

In [86]:
import torch.nn as nn
import torch

# class TudorNet(nn.Module):
#     def __init__(self, in_shape = WINDOW, hidden_state=128, layers=3, alph_size=None):
#         super(TudorNet, self).__init__()
#         self.lstm = nn.LSTM(input_size=in_shape, hidden_size=hidden_state, num_layers=layers)
#         self.fc1 = nn.Linear(in_features=hidden_state, out_features=alph_size)
    
#     def forward(self, X):
#         self.out = self.lstm(X) # (hidden, (w1, w2))
#         # print(self.out[0].shape, self.out[1][0].shape, self.out[1][1].shape)
#         self.out = self.fc1(self.out[0])
#         return self.out

class TudorNet(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_state=128, layers=3):
        super(TudorNet, self).__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_state, num_layers=layers, batch_first=True)
        self.fc1 = nn.Linear(in_features=hidden_state, out_features=vocab_size)
    
    def forward(self, X):  # X: (batch, seq_len)
        X = self.embedding(X.long())  # (batch, seq_len, embed_dim)
        out, _ = self.lstm(X)         # (batch, seq_len, hidden_size)
        out = self.fc1(out[:, -1])    # Use only the last time step's output
        return out                    # (batch, vocab_size)


In [87]:
model = TudorNet(vocab_size=total_words)

opt = torch.optim.Adam(model.parameters(), lr=2e-4)
loss_fn = nn.CrossEntropyLoss()
batch_size = 64

In [88]:
from sklearn.model_selection import train_test_split

train_X, test_X, train_y, test_y = train_test_split(X, y, shuffle=False, test_size=0.2)

In [89]:
import tqdm

def bench(seed_sent = "food"):
    
    def inferance(sent):
        tokenized_sent = []

        for word in sent.split(): tokenized_sent.append(tok.word_index[word])

        tokenized_sent = torch.Tensor([tokenized_sent])
        pred = model(tokenized_sent)

        # print(tok.index_word[pred.argmax().item()])
        return tok.index_word[pred.argmax().item()]
    for i in range(10):
        try:
            next_word = inferance(seed_sent)
        except Exception:
            print("--- Done ---")
            break;
        print(seed_sent)
        seed_sent += " " + next_word

def train(epoch: int):
    print("epoch: ", epoch)

    running_loss = 0

    model.train()

    for i in tqdm.tqdm(range(0, len(train_X), batch_size)):
        X_ = torch.Tensor(train_X[i:i + batch_size])
        y_ = torch.LongTensor(train_y[i:i + batch_size])

        opt.zero_grad()

        output = model(X_)
        loss = loss_fn(output, y_)
        running_loss += loss.item()

        loss.backward();
        opt.step();

    print("Loss: ", running_loss / len(train_X))
    model.eval()
    with torch.no_grad():
        bench()



In [90]:
for i in range(10): train(i)

epoch:  0


100%|██████████| 386/386 [00:11<00:00, 33.33it/s]


Loss:  0.12600513871254115
food
food like
food like like
food like like like
food like like like like
food like like like like like
food like like like like like like
food like like like like like like chips
food like like like like like like chips chips
food like like like like like like chips chips chips
epoch:  1


100%|██████████| 386/386 [00:10<00:00, 37.55it/s]


Loss:  0.11740093583754554
food
food like
food like like
food like like chips
food like like chips chips
food like like chips chips chips
food like like chips chips chips chips
food like like chips chips chips chips chips
food like like chips chips chips chips chips chips
food like like chips chips chips chips chips chips chips
epoch:  2


...

Loss:  0.11242933721843829
food
food like
food like like
food like like flavor
food like like flavor chips
food like like flavor chips chips
food like like flavor chips chips like
food like like flavor chips chips like like
food like like flavor chips chips like like food
food like like flavor chips chips like like food like
epoch:  8


100%|██████████| 386/386 [00:11<00:00, 34.54it/s]


Loss:  0.11177374539435096
food
food like
food like like
food like like flavor
food like like flavor chips
food like like flavor chips chips
food like like flavor chips chips like
food like like flavor chips chips like like
food like like flavor chips chips like like food
food like like flavor chips chips like like food one
epoch:  9


100%|██████████| 386/386 [00:10<00:00, 35.10it/s]

Loss:  0.11125022714964171
food
food like
food like like
food like like one
food like like one like
food like like one like chips
food like like one like chips like
food like like one like chips like like
food like like one like chips like like food
food like like one like chips like like food one


In [106]:
bench("i think that")

i think that
i think that good
i think that good great
i think that good great chips
i think that good great chips chips
i think that good great chips chips like
i think that good great chips chips like like
i think that good great chips chips like like good
i think that good great chips chips like like good flavor
i think that good great chips chips like like good flavor chips
